# Part 2 — Market Regime Classification (4-State HMM)

**Goal**: Classify every trading day (1996–2019) into one of four market regimes using a Gaussian HMM trained on:
- VIX level
- Variance Risk Premium (VRP)
- 21-day SPX momentum
- Median options bid-ask spread

**Output**: Daily regime labels, state summary statistics, transition matrix, and crisis validation table.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from features import build_daily_features
from regime import RegimeClassifier
from plots import plot_regime_timeline

DATA = '../data'

In [ ]:
# Build feature set
features = build_daily_features(
    f'{DATA}/cboeall1986.parquet',
    f'{DATA}/ff3.parquet',
    f'{DATA}/spx_clean.parquet',
)
print(f'{len(features)} trading days')
features.describe().round(3)

In [ ]:
# Fit 4-state HMM
clf = RegimeClassifier(n_states=4, random_state=42)
clf.fit(features)
state_names = clf.label_states(features)
print('State name mapping:', state_names)

In [ ]:
# State summary statistics
clf.state_summary(features)

In [ ]:
# Transition matrix
clf.transition_matrix()

In [ ]:
# Regime timeline
named_labels = clf.predict_named(features)
fig = plot_regime_timeline(features['date'], named_labels)
plt.show()

In [ ]:
# Crisis validation — do HMM states align with known events?
clf.crisis_validation(features['date'], named_labels)

In [ ]:
# Save regime labels for use in Part 3
regime_output = features[['date']].copy()
regime_output['regime'] = named_labels.values
regime_output.to_parquet(f'{DATA}/regime_labels.parquet', index=False)
print('Saved data/regime_labels.parquet')
regime_output['regime'].value_counts()

In [ ]:
# Robustness: 3-state HMM
clf3 = RegimeClassifier(n_states=3, random_state=42)
clf3.fit(features)
clf3.label_states(features)
print('3-state transition matrix:')
display(clf3.transition_matrix())
print('\n3-state crisis validation:')
display(clf3.crisis_validation(features['date'], clf3.predict_named(features)))